# Генерація Ембедінгів через XLM-RoBERTa

Використовуємо XLM-RoBERTa для отримання векторних представлень відгуків

In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from tqdm.auto import tqdm
import warnings

warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Завантаження даних

In [ ]:
df = pd.read_csv('../../data/doctors_reviews_engineered.csv')
print(f"Завантажено {len(df):,} відгуків")

# Фільтруємо тільки відгуки з текстом для ембедінгів
df_with_text = df[df['Коментар'].notna()].copy()
print(f"Відгуків з текстом: {len(df_with_text):,}")

## 2. Завантаження моделі XLM-RoBERTa

In [ ]:
MODEL_NAME = 'xlm-roberta-base'

print(f"Завантаження моделі {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model.to(device)
model.eval()
print("Модель завантажена.")

## 3. Функція для генерації ембедінгів

In [ ]:
def get_embeddings(texts, batch_size=32, max_length=128):
    """
    Генерує ембедінги для списку текстів
    
    Args:
        texts: список текстів
        batch_size: розмір батчу
        max_length: максимальна довжина токенів
        
    Returns:
        numpy array з ембедінгами
    """
    all_embeddings = []
    
    for i in tqdm(range(0, len(texts), batch_size), desc="Generating embeddings"):
        batch_texts = texts[i:i + batch_size]
        
        # Токенізація
        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        )
        
        # Переносимо на device
        encoded = {k: v.to(device) for k, v in encoded.items()}
        
        # Генеруємо ембедінги
        with torch.no_grad():
            outputs = model(**encoded)
            # Беремо [CLS] token (перший токен)
            embeddings = outputs.last_hidden_state[:, 0, :]
            
        # Переносимо на CPU та конвертуємо в numpy
        all_embeddings.append(embeddings.cpu().numpy())
        
        # Очищаємо GPU memory
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    
    return np.vstack(all_embeddings)

print("Функція для генерації ембедінгів готова")

## 4. Генерація ембедінгів для всіх відгуків

In [ ]:
# Підготовка текстів
texts = df_with_text['Коментар'].astype(str).tolist()
print(f"Генеруємо ембедінги для {len(texts):,} текстів...")

# Генеруємо ембедінги
embeddings = get_embeddings(texts, batch_size=32, max_length=128)

print(f"\nЕмбедінги згенеровано! Shape: {embeddings.shape}")
print(f"Розмірність вектора: {embeddings.shape[1]}")

## 5. Збереження ембедінгів

In [ ]:
# Зберігаємо ембедінги як numpy array
np.save('../../data/xlm_roberta_embeddings.npy', embeddings)
print(f"Ембедінги збережено: ../../data/xlm_roberta_embeddings.npy")
print(f"Розмір файлу: {embeddings.nbytes / (1024**2):.2f} MB")

In [ ]:
# Зберігаємо індекси відгуків з текстом (щоб потім звязати з основним датасетом)
text_indices = df_with_text.index.tolist()
np.save('../../data/text_indices.npy', text_indices)
print(f"Індекси збережено: {len(text_indices):,} записів")

## 6. Додаткові ознаки з ембедінгів

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Для кожного відгуку обчислюємо середню подібність з іншими відгуками того ж лікаря
print("Обчислюємо подібність ознак...")

similarity_features = []

for doctor in tqdm(df_with_text["Ім'я лікаря"].unique(), desc="Обробка лікаріів"):
    doctor_mask = df_with_text["Ім'я лікаря"] == doctor
    doctor_indices = df_with_text[doctor_mask].index
    
    if len(doctor_indices) < 2:
        continue
    
    # Отримуємо ембедінги для цього лікаря
    doctor_embeddings_indices = [list(text_indices).index(idx) for idx in doctor_indices if idx in text_indices]
    doctor_embeddings = embeddings[doctor_embeddings_indices]
    
    # Обчислюємо cosine similarity matrix
    sim_matrix = cosine_similarity(doctor_embeddings)
    
    # Для кожного відгуку - середня similarity з іншими (виключаючи себе)
    for i, idx in enumerate(doctor_indices):
        if idx in text_indices:
            # Середня similarity з іншими відгуками (виключаючи діагональ)
            sim_scores = sim_matrix[i]
            avg_sim = (sim_scores.sum() - 1) / (len(sim_scores) - 1) if len(sim_scores) > 1 else 0
            max_sim = sim_scores[sim_scores < 1].max() if len(sim_scores) > 1 else 0
            
            similarity_features.append({
                'index': idx,
                'avg_similarity': avg_sim,
                'max_similarity': max_sim
            })

print(f"Similarity features обчислено для {len(similarity_features):,} відгуків")

In [ ]:
# Конвертуємо в DataFrame і зберігаємо
if similarity_features:
    sim_df = pd.DataFrame(similarity_features)
    sim_df.to_csv('../../data/similarity_features.csv', index=False)
    print(f"Similarity features збережено: {len(sim_df):,} записів")
    print("\nСтатистика:")
    print(sim_df[['avg_similarity', 'max_similarity']].describe())